# Document Ingestion Pipeline: PDF Extraction & Document Understanding

## 1. Overview
Document ingestion transforms unstructured, layout-rich formats (PDFs, forms, invoices) into clean, semantically structured text and tables suitable for chunking, embedding, and RAG retrieval.

### Pipeline Stages Covered:
1. **Scanned vs. Native Detection**: Classify PDF pages by selectable text density and route scanned pages to dedicated OCR.
2. **Fast Markdown & Table Extraction with PyMuPDF4LLM**: Parse multi-column text, reading order, and tabular structures without heavy deep-learning overhead.
3. **Interactive Form Fields**: Extract strongly typed key-value pairs from AcroForm and interactive PDF fields using PyMuPDF.
4. **Batch Processing**: Scalable batch conversion pipelines exported to HuggingFace datasets.
5. **Structured Extraction (NER)**: spaCy entity detection, rule-based matching and custom NER training. Template-based receipt extraction lives in `6.3-Ingestion-OCR.ipynb`.

> [!NOTE]
> For scanned image-only documents requiring image preprocessing, Tesseract layout reconstruction, or Vision-Language Models (VLMs), refer to [**`6.3-Ingestion-OCR.ipynb`**](6.3-Ingestion-OCR.ipynb).


---
## 1. Advanced PDF Extraction & Document Understanding (PyMuPDF & PyMuPDF4LLM)


In [ ]:
%pip install pymupdf pymupdf4llm datasets tqdm rich requests spacy -q


In [ ]:
import requests
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Reuse attention.pdf from 3.3 if available, else re-download
PDF_PATH = DATA_DIR / "attention.pdf"
if not PDF_PATH.exists():
    print("Downloading attention.pdf...")
    r = requests.get("https://arxiv.org/pdf/1706.03762", timeout=30)
    PDF_PATH.write_bytes(r.content)

print(f"PDF ready: {PDF_PATH} ({PDF_PATH.stat().st_size / 1024:.1f} KB)")

## 1. Detecting Scanned vs. Text Pages

Before OCR, automatically classify each page as text-based or scanned (image-based).

In [ ]:
import fitz

def classify_pages(pdf_path: Path, text_threshold: int = 50) -> list[dict]:
    """Classify each page as 'text' or 'scanned' based on extracted character count."""
    doc = fitz.open(pdf_path)
    results = []
    for i, page in enumerate(doc):
        text = page.get_text().strip()
        images = page.get_images()
        page_type = "text" if len(text) >= text_threshold else "scanned"
        results.append({
            "page": i + 1,
            "type": page_type,
            "char_count": len(text),
            "image_count": len(images),
        })
    return results

page_info = classify_pages(PDF_PATH)
scanned = [p for p in page_info if p["type"] == "scanned"]
text_pages = [p for p in page_info if p["type"] == "text"]

print(f"Total pages : {len(page_info)}")
print(f"Text pages  : {len(text_pages)}")
print(f"Scanned     : {len(scanned)}")
print("\nPage breakdown (first 10):")
for p in page_info[:10]:
    print(f"  Page {p['page']:3d} | {p['type']:8s} | chars={p['char_count']:5d} | images={p['image_count']}")

### Routing Scanned Pages to Dedicated OCR Pipeline

When a PDF page is classified as `scanned` (insufficient selectable text characters), it cannot be parsed by standard text extractors and requires an optical character recognition (OCR) pipeline.

> [!NOTE]
> For full OCR preprocessing, Tesseract configuration, word bounding boxes, layout reconstruction, and multimodal Vision-Language Models (VLMs), refer to the dedicated module: [**`6.3-Ingestion-OCR.ipynb`**](6.3-Ingestion-OCR.ipynb).

For digitally generated and native text PDFs, we proceed directly with AI-driven layout and structure extraction using **PyMuPDF4LLM** below.


## 2. Fast Structure & Markdown Extraction with PyMuPDF4LLM

`pymupdf4llm` provides fast, lightweight layout-aware text extraction directly into GitHub-flavored Markdown. It identifies headings, paragraphs, lists, bold text, code blocks, and markdown tables without requiring external heavy deep-learning models or GPU runtime.


In [ ]:
import pymupdf4llm

# Convert PDF to structured Markdown
md_pymupdf = pymupdf4llm.to_markdown(str(PDF_PATH))

print("Extraction complete!")
print(f"Markdown length: {len(md_pymupdf):,} characters")
print("\n--- First 1,500 characters ---")
print(md_pymupdf[:1500])


In [ ]:
import re

# Count structural elements in the extracted markdown
h1_tags = re.findall(r"^# (.+)$", md_pymupdf, re.MULTILINE)
h2_tags = re.findall(r"^## (.+)$", md_pymupdf, re.MULTILINE)
h3_tags = re.findall(r"^### (.+)$", md_pymupdf, re.MULTILINE)
table_rows = re.findall(r"^\|", md_pymupdf, re.MULTILINE)
bold_spans = re.findall(r"\*\*.+?\*\*", md_pymupdf)

print("Document Structure Summary:")
print(f"  Total characters : {len(md_pymupdf):,}")
print(f"  Level 1 Headings : {len(h1_tags)}")
print(f"  Level 2 Headings : {len(h2_tags)}")
print(f"  Level 3 Headings : {len(h3_tags)}")
print(f"  Table rows       : {len(table_rows)}")
print(f"  Bold emphasis    : {len(bold_spans)}")

print("\nTop-level Sections Detected:")
for h in h1_tags[:8]:
    print(f"  • {h}")


In [ ]:
import fitz

# Extract tables natively using PyMuPDF TableFinder
doc = fitz.open(PDF_PATH)
tables_found = []

for page_idx, page in enumerate(doc):
    tab_finder = page.find_tables()
    for tab in tab_finder.tables:
        tables_found.append((page_idx + 1, tab.extract()))

print(f"Tables detected across document: {len(tables_found)}")

# Preview first extracted table
if tables_found:
    page_num, first_tab = tables_found[0]
    print(f"\n=== First Table (Page {page_num}) ===\n")
    for row in first_tab[:5]:
        print("  | " + " | ".join(str(cell or "").strip() for cell in row) + " |")


## 3. Extracting Form Fields

PDF forms have interactive fields (text inputs, checkboxes, dropdowns). PyMuPDF can read and modify them.

In [ ]:
# Create a minimal PDF form for demonstration
def create_sample_form(output_path: Path):
    """Create a simple PDF with fillable form fields using PyMuPDF."""
    doc = fitz.open()
    page = doc.new_page()

    fields = [
        ("Name:",    fitz.Rect(120, 100, 400, 120), "text",     "John Doe"),
        ("Email:",   fitz.Rect(120, 140, 400, 160), "text",     "john@example.com"),
        ("Score:",   fitz.Rect(120, 180, 200, 200), "text",     "95"),
    ]

    page.insert_text((50, 60), "Sample NLP Evaluation Form", fontsize=16)

    for label, rect, ftype, value in fields:
        page.insert_text((50, rect.y0 + 12), label, fontsize=11)
        widget = fitz.Widget()
        widget.rect = rect
        widget.field_type = fitz.PDF_WIDGET_TYPE_TEXT
        widget.field_name = label.strip(":")
        widget.field_value = value
        widget.border_color = (0, 0, 0)
        widget.fill_color = (0.95, 0.95, 1.0)
        page.add_widget(widget)

    doc.save(str(output_path))
    print(f"Form saved: {output_path}")

FORM_PATH = DATA_DIR / "sample_form.pdf"
create_sample_form(FORM_PATH)

In [ ]:
# Extract all form field values from a PDF
def extract_form_fields(pdf_path: Path) -> list[dict]:
    doc = fitz.open(pdf_path)
    fields = []
    for page_num, page in enumerate(doc):
        for widget in page.widgets():
            fields.append({
                "page":        page_num + 1,
                "field_name":  widget.field_name,
                "field_type":  widget.field_type_string,
                "value":       widget.field_value,
                "rect":        list(widget.rect),
            })
    return fields

fields = extract_form_fields(FORM_PATH)
print(f"Form fields found: {len(fields)}")
for f in fields:
    print(f"  [{f['field_type']:10s}] {f['field_name']:20s} = {f['value']}")

## 4. Batch PDF Processing Pipeline

Process a folder of PDFs — auto-detect page types, extract with the right method, save as a HuggingFace `Dataset`.

In [ ]:
# Download a small set of papers for batch processing
BATCH_DIR = DATA_DIR / "papers"
BATCH_DIR.mkdir(exist_ok=True)

papers = {
    "attention.pdf": "https://arxiv.org/pdf/1706.03762",
    "bert.pdf":      "https://arxiv.org/pdf/1810.04805",
    "gpt2.pdf":      "https://arxiv.org/pdf/2005.14165",  # GPT-3 paper
}

for filename, url in papers.items():
    dest = BATCH_DIR / filename
    if not dest.exists():
        print(f"Downloading {filename}...")
        try:
            r = requests.get(url, timeout=30)
            dest.write_bytes(r.content)
            print(f"  {filename}: {dest.stat().st_size/1024:.1f} KB")
        except Exception as e:
            print(f"  Failed: {e}")
    else:
        print(f"  {filename} already exists")

In [ ]:
import re
from datetime import datetime

import fitz
from tqdm import tqdm

def process_pdf_for_dataset(pdf_path: Path) -> dict | None:
    """Extract a PDF into a dataset record with metadata, markdown, and chunks."""
    try:
        # Use pymupdf4llm for fast Markdown extraction
        md = pymupdf4llm.to_markdown(str(pdf_path))

        # Extract metadata natively via PyMuPDF
        doc = fitz.open(pdf_path)
        meta = doc.metadata or {}
        page_count = len(doc)
        doc.close()

        # Extract title from first H1 in markdown or metadata
        title = ""
        for line in md.split("\n"):
            if line.startswith("# "):
                title = line.lstrip("# ").strip()
                break

        # Semantic chunks by heading
        sections = re.split(r"\n(?=#{1,3} )", md)
        chunks = [{"index": i, "text": s.strip()} for i, s in enumerate(sections) if s.strip()]

        return {
            "file":         pdf_path.name,
            "title":        title or meta.get("title") or pdf_path.stem,
            "author":       meta.get("author", ""),
            "pages":        page_count,
            "extracted_at": datetime.now().isoformat(),
            "markdown":     md,
            "num_chunks":   len(chunks),
            "chunks":       [c["text"] for c in chunks],
        }
    except Exception as e:
        print(f"  ERROR on {pdf_path.name}: {e}")
        return None

# Run batch
pdf_files = sorted(BATCH_DIR.glob("*.pdf"))
records = []
for pdf in tqdm(pdf_files, desc="Processing PDFs"):
    record = process_pdf_for_dataset(pdf)
    if record:
        records.append(record)
        print(f"  {record['file']:20s} | pages={record['pages']:3d} | chunks={record['num_chunks']}")

print(f"\nProcessed {len(records)} / {len(pdf_files)} PDFs successfully")


In [ ]:
# Save as HuggingFace Dataset
from datasets import Dataset

# Build flat chunk-level dataset (better for training)
chunk_records = []
for record in records:
    for i, chunk_text in enumerate(record["chunks"]):
        if len(chunk_text.strip()) > 50:  # skip trivial chunks
            chunk_records.append({
                "text":   chunk_text,
                "source": record["file"],
                "title":  record["title"],
                "chunk":  i,
                "pages":  record["pages"],
            })

ds = Dataset.from_list(chunk_records)
print(ds)
print(f"\nTotal chunks: {len(ds)}")
print(f"\nSample record:")
print(f"  source: {ds[0]['source']}")
print(f"  title:  {ds[0]['title']}")
print(f"  text:   {ds[0]['text'][:200]}...")

In [ ]:
# Save the dataset locally
ds_path = DATA_DIR / "papers_dataset"
ds.save_to_disk(str(ds_path))
print(f"Dataset saved to: {ds_path}")

# Reload and verify
from datasets import load_from_disk
ds_loaded = load_from_disk(str(ds_path))
print(f"Reloaded: {ds_loaded}")

# Also save as JSONL for maximum compatibility
jsonl_path = DATA_DIR / "papers_chunks.jsonl"
ds.to_json(str(jsonl_path))
print(f"JSONL saved: {jsonl_path} ({jsonl_path.stat().st_size / 1024:.1f} KB)")

## 5. Exercise — Build Your Own Extraction Pipeline

Pick any PDF (a textbook chapter, a legal document, a research paper, a scanned form) and complete the following:

1. **Detect** whether each page is text-based or scanned
2. **Extract** using the appropriate method (native text extraction vs. routing to OCR in 6.4)
3. **Parse structure** — title, sections, tables
4. **Chunk** using heading-based strategy
5. **Export** as a HuggingFace `Dataset` and inspect it

Stretch goal: test table extraction with `page.find_tables()` on a PDF with complex tables and inspect the extracted markdown.

In [ ]:
# Your code here
YOUR_PDF = DATA_DIR / "your_document.pdf"  # replace with your PDF path

# Step 1: classify pages
# Step 2: extract text
# Step 3: parse structure
# Step 4: chunk
# Step 5: export as Dataset

---
## 5. Structured Extraction: Named Entities (spaCy)

Beyond layout, ingestion often needs **entities**: people, organizations, places, amounts. [spaCy](https://spacy.io/) gives entity detection out of the box, rule-based matching, and a path to training your own tagger. Entity tags later become metadata filters in the vector store.

In [ ]:
import subprocess, sys
import spacy

try:
    spacy.load("en_core_web_lg")
except OSError:   # download the model once
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_lg"])

In [ ]:
import spacy
from spacy import displacy

In [ ]:
#smaller model
#nlp = spacy.load("en_core_web_sm")

#larger model
nlp = spacy.load("en_core_web_lg")

### 5.1 Entity Detection

In [ ]:
# document level entities
def detectEntities(doc):   
    for ent in doc.ents:
        print(ent.text, ent.start_char, ent.end_char, ent.label_)
    print()
    displacy.render(doc, style='ent', jupyter=True)

In [ ]:
# document level sentences
def detectSentences(doc):   
    for sent in doc.sents:
        print("Sentence: ", sent)

In [ ]:
#geopolitical entity
doc = nlp(u'San Francisco considers banning sidewalk delivery robots')
detectEntities (doc)

In [ ]:
#organization
doc = nlp(u'Apple is looking at buying U.K. startup for $1 billion. Google might make their own offer.')
detectEntities (doc)
detectSentences (doc)

In [ ]:
#some ambiguities
doc = nlp(u'Amazon is the largest forest in Brazil. Amazon is the largest retailer in the world.')
detectEntities (doc)

In [ ]:
text = "Recently, Amazon announced an investment of $2 billion in renewable energy projects across Asia. Warren Buffett's Berkshire Hathaway increased its stake in Apple by purchasing $3 billion worth of shares. Elon Musk has committed $1.5 billion towards SpaceX's new satellite communications infrastructure. Bill Gates, co-chair of the Bill & Melinda Gates Foundation, donated $500 million to global health initiatives combating malaria."
doc = nlp(text)
detectSentences (doc)
detectEntities(doc)

### 5.2 NER Tagging

#### 5.2.1 Rule-based NER

In [ ]:
import spacy
from spacy.matcher import Matcher

# Initialize the Matcher with the shared vocabulary
matcher = Matcher(nlp.vocab)

# Define different patterns for the matcher
patterns = {
    "TITLE_NAME": [{'TEXT': 'Mr'}, {'POS': 'PROPN'}, {'POS': 'PROPN'}],  # Matches "Mr. John Smith"
    "DATE": [{'SHAPE': 'Xxxxx'}, {'IS_DIGIT': True, 'LENGTH': {'==': 1}}, {'IS_DIGIT': True, 'LENGTH': {'==': 4}}],  # Matches "January 1, 2021"
    "MONEY": [{'TEXT': '$'}, {'IS_DIGIT': True}]  # Matches "$100"
}

# Add patterns to the matcher
for label, pattern in patterns.items():
    matcher.add(label, [pattern])

# Processing text
doc = nlp("Mr. John Smith paid $100 on January 1, 2021.")

# Apply the matcher to the doc
matches = matcher(doc)
print("Matches found:")
for match_id, start, end in matches:
    span = doc[start:end]
    print(f"{span.text} - {nlp.vocab.strings[match_id]}")  # Display match and label


#### 5.2.2 Train Machine Learning Model 

In [ ]:
import spacy
from spacy.training import Example

# Load a pre-existing spaCy model
nlp = spacy.blank('en')  # Start with a blank English model

# Create a new NER component and add it to the pipeline if it doesn't exist
if 'ner' not in nlp.pipe_names:
    ner = nlp.add_pipe('ner', last=True)
else:
    ner = nlp.get_pipe('ner')

# Add a new entity label to the NER component
ner.add_label('TECHNOLOGY')

# Training data: text with entities annotated
TRAIN_DATA = [
    ("Apple releases new products every year.", {"entities": [(0, 5, "TECHNOLOGY")]}),
    ("Microsoft developed Windows 11.", {"entities": [(0, 9, "TECHNOLOGY")]}),
]

# Set up the training loop
optimizer = nlp.initialize()

# Training loop
for i in range(10):
    for text, annotations in TRAIN_DATA:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        nlp.update([example], drop=0.5, sgd=optimizer)

# Test the trained model
test_text = "Google and Apple are leading in the technology sector."
doc = nlp(test_text)
for ent in doc.ents:
    print(f"{ent.text} ({ent.label_})")

#### 5.2.3  Generating Synthetic Data

In [ ]:
import random

entities = ['Google', 'Amazon', 'Facebook']
templates = ['[ENTITY] has announced a new product.', '[ENTITY] is now hiring.', '[ENTITY] just released their quarterly results.']

def generate_synthetic_data(entities, templates, n=10):
    synthetic_data = []
    for _ in range(n):
        entity = random.choice(entities)
        template = random.choice(templates)
        sentence = template.replace('[ENTITY]', entity)
        synthetic_data.append(sentence)
    return synthetic_data

# Generate synthetic sentences
synthetic_sentences = generate_synthetic_data(entities, templates, 100)
for sentence in synthetic_sentences:
    print(sentence)


## Summary & Key Takeaways

1. **Document Fidelity**: Clean, structure-preserving extraction directly determines downstream embedding quality and retrieval precision.
2. **Dynamic vs. Static**: Use BeautifulSoup/Trafilatura for high-speed static HTML extraction (50x faster); reserve Playwright for client-rendered JavaScript SPAs.
3. **Hybrid OCR Routing**: Route digitally generated PDFs to PyMuPDF / PyMuPDF4LLM (fast, exact text) and route scanned image pages to the dedicated OCR pipeline in `6.3-Ingestion-OCR.ipynb`.
4. **Entities as Metadata**: NER tags (people, organizations, places) become filterable metadata and feed entity-aware indexing.
5. **Metadata Preservation**: Always retain document headers, source URLs, page numbers, and author timestamps to enable filtered search in vector databases.